In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [10]:
import requests
from bs4 import BeautifulSoup, element
import pandas as pd
from datetime import datetime 

In [280]:
# -----------------------
# variable
# -----------------------
headers = {
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'
}

NO_DATA: str = ""

DATE: str = "날짜"
CLOSE: str = "종가"
OPEN: str = "시가"
HIGH: str = "고가"
LOW: str = "저가"
VOLUME: str = "거래량"
DIFF: str = "전일비"
COLUMNS = [DATE, CLOSE, DIFF, OPEN, HIGH, LOW, VOLUME]
# COLUMNS = ["날짜", "종가", "전일비", "시가", "고가", "저가", "거래량"]


# -----------------------
# function
# -----------------------
def request_day_sise_to_naver(arg_code: str, arg_page: int) -> BeautifulSoup:
    """
    naver 에 주식 일일 시세를 조회한다.\n
    함수 내부에 url 을 갖고 있다.\n
    종목 코드와 페이지 번호를 인자로 받기 때문에 모든 일일 시세를 확보할 수 있다.\n

    :param arg_code: 6자리 종목 코드
    :param arg_page: 페이지 번호
    :return: BeautifulSoup 객체
    """
    url = "https://finance.naver.com/item/sise_day.naver"
    ret = requests.get(url="{}?code={}&page={}".format(url, arg_code, arg_page), headers=headers)
    if ret.status_code == 200:
        return BeautifulSoup(ret.text, 'html.parser')


def get_tr_list(arg_soup: BeautifulSoup) -> element.ResultSet:
    """
    첫 번째 table 로부터 tr 들을 검색하여 반환한다.\n
    table 과 tr 선택 기준은 태그의 속성에 따른다.\n
    속성 정보는 본 함수의 변수에 문자열 상수로 담겨 있다.\n

    :param arg_soup: table 데이터가 담긴 BeautifulSoup
    :return: tr 태그들을 담은 ResultSet
    """
    table_attr_class = "type2"
    tr_attr_dict_key = "onmouseover"
    tr_attr_dict_value = "mouseOver(this)"

    # ret = arg_soup.select_one("table.{}".format(table_attr_class)) \
    #     .select("tr[{}='{}']".format(tr_attr_dict_key, tr_attr_dict_value))
    ret = arg_soup.find("table", class_=table_attr_class) \
        .find_all("tr", attrs={tr_attr_dict_key: tr_attr_dict_value})
    return ret


def get_td_list_from_tr_list(arg_list: list, tr_elements: element.ResultSet):
    """
    인자로 tr 목록이 주어진다.(tr_elements: bs4.element.ResultSet)\n
    각각의 tr 에 대하여 td 들을 뽑아낸다.(bs4.element.Tag)\n
    td 태그의 value 들을 모아서 list 를 만든다.(list)\n
    td 태그의 value 중에 하나라도 "" 가 나타나면 함수를 종료한다.\n
    list 를 arg_list 에 담는다.(list[list])\n

    :param arg_list: td 의 value(text) 를 모은 list 를 담을 list
    :param tr_elements: td 를 검색할 대상 tr 목록
    :return: td 의 value(text) 가 "" 일 경우 함수 종료
    """
    for tr in tr_elements:
        td_elements = tr.select("td")
        ret = []
        for td in td_elements:
            td_data = td.text.strip()
            if td_data == "":
                return
            ret.append(td_data)
        arg_list.append(ret)


def convert_naver_day_sise(arg_list: list):
    """
    주어진 행렬(이중 list) 에서 데이터를 하나씩 꺼내어 변환 후 다시 저장한다.\n
    첫 번째 데이터는 날짜 타입으로 변환하고 그 외의 것들은 숫자 타입으로 변환한다.\n
    날짜 타입으로 변환할 데이터는 "2022.03.15" 와 같은 날짜 정보의 문자열이다.\n
    숫자 타입으로 변환할 데이터는 "33,336" 와 같이 쉼표를 포함한 숫자들의 문자열이다.\n

    :param arg_list: 행렬(이중 list)
    :return:
    """
    l1: list
    for l1 in arg_list:
        for i, d1 in enumerate(l1):
            # ------------------------------
            # 첫 번째 데이터는 날짜 타입으로 변환
            # ------------------------------
            if i == 0:
                l1[i] = datetime.strptime(d1, "%Y.%m.%d")
            else:
                l1[i] = int(d1.replace(",", ""))


def get_key_list(arg_list: list, arg_i: int) -> set:
    """
    주어진 행렬(이중 list)에서 특정 컬럼의 값들을 뽑아내어 set 으로 반환한다.\n
    특정 컬럼의 값들을 기준으로 주어진 행렬과 다른 행렬이 같은지 다른지 확인한다.\n

    :param arg_list: 행렬(이중 list)
    :param arg_i: 선택하려는 컬럼의 index
    :return: 선택한 컬럼들을 모아 생성한 set
    """
    ret = set()
    for l1 in arg_list:
        ret.add(l1[arg_i])
    return ret


def get_days_sise_by_page(arg_code: str, arg_page: int) -> pd.DataFrame:
    """
    naver 에 일일 시세(특정 종목의 특정 페이지)를 조회하여 list 로 생성 및 반환한다.\n

    :param arg_code: 조회하려는 종목의 코드 6자리
    :param arg_page: 조회하려는 페이지
    :return: 시세 종목을 담은 pd.DataFrame
    """
    # 총 10일치의 데이터
    ret_list = []
    soup = request_day_sise_to_naver(arg_code, arg_page)
    tr_list = get_tr_list(soup)
    get_td_list_from_tr_list(ret_list, tr_list)
    convert_naver_day_sise(ret_list)
    ret = pd.DataFrame(ret_list, columns=COLUMNS)
    return ret


def load_data(arg_filename: str) -> pd.DataFrame:
    """
    시세 데이터가 담겨 있는 csv 파일을 읽어서 DataFrame 을 생성 반환한다.
    DATE 컬럼을 datetime 타입으로 변환한다.
    datetime 타입을 파일에 기록하면 str 으로 변환되기 때문이다.
    만약 DATE 컬럼에 대하여 오름차순으로 정렬한다.

    :param arg_filename: 읽고자 하는 파일의 이름
    :return: 파일의 내용이 담긴 DataFrame 객체
    """
    df = pd.read_csv(arg_filename)
    df[DATE] = pd.to_datetime(df[DATE])
    return df.sort_values(by=DATE, ascending=True, na_position='last')
    
    
def save_data(arg_filename: str, arg_df: pd.DataFrame) -> None:
    """
    DataFrame 의 index 를 DATE 로 변경한다.
    DataFrame 을 파일에 저장한다.
    
    :param arg_filename: DataFrame 을 저장할 파일의 이름
    :param arg_df: 파일에 저장할 DataFrame 객체
    :return: 
    """
    if arg_df.index.name is None:
        arg_df = arg_df.set_index(DATE)
    elif arg_df.index.name != DATE:
        arg_df = arg_df.reset_index().set_index(DATE)
    arg_df.sort_values(by=DATE, ascending=True, na_position='last').to_csv(arg_filename)
    

---

- # DataFrame.set_index()
- # DataFrame.reset_index()
- # DataFrame.dtypes()

In [233]:
code = "005930"
df = get_days_sise_by_page(code, 1)
df
df.dtypes

,날짜,종가,전일비,시가,고가,저가,거래량
0,2022-07-01,56200,800,56900,57500,55900,24613445
1,2022-06-30,57000,1000,57200,57600,57000,18915142
2,2022-06-29,58000,1400,58500,58800,58000,14677138
3,2022-06-28,59400,600,59200,59500,58700,13540538
4,2022-06-27,58800,400,59000,59900,58300,18122236
5,2022-06-24,58400,1000,57900,59100,57700,23256103
6,2022-06-23,57400,200,57700,58000,56800,28338608
7,2022-06-22,57600,900,59000,59100,57600,23334687
8,2022-06-21,58500,200,58700,59200,58200,25148109
9,2022-06-20,58700,1100,59800,59900,58100,34111306


날짜     datetime64[ns]
종가              int64
전일비             int64
시가              int64
고가              int64
저가              int64
거래량             int64
dtype: object

In [234]:
df1 = df.set_index(DATE)
df1.dtypes
df1.head()

종가     int64
전일비    int64
시가     int64
고가     int64
저가     int64
거래량    int64
dtype: object

,종가,전일비,시가,고가,저가,거래량
날짜,,,,,,
2022-07-01,56200,800,56900,57500,55900,24613445
2022-06-30,57000,1000,57200,57600,57000,18915142
2022-06-29,58000,1400,58500,58800,58000,14677138
2022-06-28,59400,600,59200,59500,58700,13540538
2022-06-27,58800,400,59000,59900,58300,18122236


In [291]:
df2 = df1.reset_index()
df2
df2.dtypes

,날짜,종가,전일비,시가,고가,저가,거래량
0,2022-07-01,56200,800,56900,57500,55900,24613445
1,2022-06-30,57000,1000,57200,57600,57000,18915142
2,2022-06-29,58000,1400,58500,58800,58000,14677138
3,2022-06-28,59400,600,59200,59500,58700,13540538
4,2022-06-27,58800,400,59000,59900,58300,18122236
5,2022-06-24,58400,1000,57900,59100,57700,23256103
6,2022-06-23,57400,200,57700,58000,56800,28338608
7,2022-06-22,57600,900,59000,59100,57600,23334687
8,2022-06-21,58500,200,58700,59200,58200,25148109
9,2022-06-20,58700,1100,59800,59900,58100,34111306


날짜     datetime64[ns]
종가              int64
전일비             int64
시가              int64
고가              int64
저가              int64
거래량             int64
dtype: object

---

# 분명 00 데이터의 타입은 Timestamp 였는데, 파일에 쓰고 다시 읽으면 str 이 된다.
- # DataFrame.iloc[ ]
- # DataFrame.iloc[ ][ ]

In [236]:
d00 = df2.iloc[0][DATE]
type(d00)
d00

pandas._libs.tslibs.timestamps.Timestamp

Timestamp('2022-07-01 00:00:00')

In [237]:
file_name: str = "{}.csv".format(code)
file_name

'005930.csv'

In [274]:
# 날짜에 대하여 내리차순으로 되어 있는 데이터를 오름차순으로 역순 정렬
# 날짜를 인덱스로 지정
# 파일로 저장

save_data(file_name, df2)

In [278]:
df3 = load_data(file_name)
df3.head()

,날짜,종가,전일비,시가,고가,저가,거래량
0,2022-06-20,58700,1100,59800,59900,58100,34111306
1,2022-06-21,58500,200,58700,59200,58200,25148109
2,2022-06-22,57600,900,59000,59100,57600,23334687
3,2022-06-23,57400,200,57700,58000,56800,28338608
4,2022-06-24,58400,1000,57900,59100,57700,23256103


In [ ]:
page = 2
df21 = get_days_sise_by_page(code, page)
# df21 = get_days_sise_by_page(code, page).iloc[::-1]
df21
type(df21.iloc[0][DATE])

,날짜,종가,전일비,시가,고가,저가,거래량
0,2022-06-17,59800,1100,59400,59900,59400,29053450
1,2022-06-16,60900,200,61300,61800,60500,23394895
2,2022-06-15,60700,1200,61300,61500,60200,26811224
3,2022-06-14,61900,200,61200,62200,61100,24606419
4,2022-06-13,62100,1700,62400,62800,62100,22157816
5,2022-06-10,63800,1400,64000,64400,63800,22193552
6,2022-06-09,65200,100,65100,65200,64500,25790725
7,2022-06-08,65300,200,65400,65700,65300,12483180
8,2022-06-07,65500,1300,66200,66400,65400,19355755
9,2022-06-03,66800,100,67200,67300,66800,8222883


pandas._libs.tslibs.timestamps.Timestamp

In [294]:
tmp = df21.sort_values(by=DATE, ascending=True, na_position='last')
last_date = tmp.iloc[0][DATE]
last_date
type(last_date)

exist1 = len(df21[df2[DATE] == last_date])
exist1

exist2 = len(df21[df21[DATE] == last_date])
exist2

Timestamp('2022-06-03 00:00:00')

pandas._libs.tslibs.timestamps.Timestamp

0

1

In [206]:
total_df = pd.concat([df21.iloc[::-1], df4])
total_df

,날짜,종가,전일비,시가,고가,저가,거래량
9,2022-06-03 00:00:00,66800,100,67200,67300,66800,8222883
8,2022-06-07 00:00:00,65500,1300,66200,66400,65400,19355755
7,2022-06-08 00:00:00,65300,200,65400,65700,65300,12483180
6,2022-06-09 00:00:00,65200,100,65100,65200,64500,25790725
5,2022-06-10 00:00:00,63800,1400,64000,64400,63800,22193552
4,2022-06-13 00:00:00,62100,1700,62400,62800,62100,22157816
3,2022-06-14 00:00:00,61900,200,61200,62200,61100,24606419
2,2022-06-15 00:00:00,60700,1200,61300,61500,60200,26811224
1,2022-06-16 00:00:00,60900,200,61300,61800,60500,23394895
0,2022-06-17 00:00:00,59800,1100,59400,59900,59400,29053450


In [215]:
total_df[DATE]

9    2022-06-03 00:00:00
8    2022-06-07 00:00:00
7    2022-06-08 00:00:00
6    2022-06-09 00:00:00
5    2022-06-10 00:00:00
4    2022-06-13 00:00:00
3    2022-06-14 00:00:00
2    2022-06-15 00:00:00
1    2022-06-16 00:00:00
0    2022-06-17 00:00:00
0             2022-06-20
1             2022-06-21
2             2022-06-22
3             2022-06-23
4             2022-06-24
5             2022-06-27
6             2022-06-28
7             2022-06-29
8             2022-06-30
9             2022-07-01
Name: 날짜, dtype: object

In [218]:
total_df[DATE] = pd.to_datetime(total_df[DATE])
type(total_df.iloc[0][DATE])

pandas._libs.tslibs.timestamps.Timestamp

In [219]:
total_df

,날짜,종가,전일비,시가,고가,저가,거래량
9,2022-06-03,66800,100,67200,67300,66800,8222883
8,2022-06-07,65500,1300,66200,66400,65400,19355755
7,2022-06-08,65300,200,65400,65700,65300,12483180
6,2022-06-09,65200,100,65100,65200,64500,25790725
5,2022-06-10,63800,1400,64000,64400,63800,22193552
4,2022-06-13,62100,1700,62400,62800,62100,22157816
3,2022-06-14,61900,200,61200,62200,61100,24606419
2,2022-06-15,60700,1200,61300,61500,60200,26811224
1,2022-06-16,60900,200,61300,61800,60500,23394895
0,2022-06-17,59800,1100,59400,59900,59400,29053450


# DateFrame.concat([]) 은 중복된 항목은 어떻게 처리할까?
# 항목들이 중복되는지 확인하지 않고 그냥 합친다.

In [223]:
df31 = get_days_sise_by_page(code, 2)
df31

,날짜,종가,전일비,시가,고가,저가,거래량
0,2022-06-17,59800,1100,59400,59900,59400,29053450
1,2022-06-16,60900,200,61300,61800,60500,23394895
2,2022-06-15,60700,1200,61300,61500,60200,26811224
3,2022-06-14,61900,200,61200,62200,61100,24606419
4,2022-06-13,62100,1700,62400,62800,62100,22157816
5,2022-06-10,63800,1400,64000,64400,63800,22193552
6,2022-06-09,65200,100,65100,65200,64500,25790725
7,2022-06-08,65300,200,65400,65700,65300,12483180
8,2022-06-07,65500,1300,66200,66400,65400,19355755
9,2022-06-03,66800,100,67200,67300,66800,8222883


1. 기존의 데이터를 불러온다.
2. 마지막 날짜의 데이터를 찾는다.
3. 새로운 데이터를 불러온다.
4. 새로운 데이터에 내가 갖고 있던 마지막 데이터가 존재하는지 확인한다.
5. 새로운 데이터가 있으면 중지, 없으면 진행한다.


In [227]:
total_df.iloc[0][DATE]

Timestamp('2022-06-03 00:00:00')

In [224]:
test_df = pd.concat([df31.iloc[::-1], total_df])
test_df

,날짜,종가,전일비,시가,고가,저가,거래량
9,2022-06-03,66800,100,67200,67300,66800,8222883
8,2022-06-07,65500,1300,66200,66400,65400,19355755
7,2022-06-08,65300,200,65400,65700,65300,12483180
6,2022-06-09,65200,100,65100,65200,64500,25790725
5,2022-06-10,63800,1400,64000,64400,63800,22193552
4,2022-06-13,62100,1700,62400,62800,62100,22157816
3,2022-06-14,61900,200,61200,62200,61100,24606419
2,2022-06-15,60700,1200,61300,61500,60200,26811224
1,2022-06-16,60900,200,61300,61800,60500,23394895
0,2022-06-17,59800,1100,59400,59900,59400,29053450
